Use this notebook to manually validate edit personal information. 
Copy this and fill with the relevant data
DO NOT SAVE THE COPY SOMEWHERE GIT CAN SEE

In [ ]:
import os
print(os.getcwd())
import sys
sys.path.append(os.path.abspath('..'))  # Adds project root if notebook is inside notebooks/
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
# from LegacyPipe import cleaning_funcs
import legacy_pipe.cleaning_funcs as cf
import legacy_pipe.ValidateID as vid
import tqdm
import pandas as pd

# Load your data
edited_df = pd.read_csv("/path/to/initial/metadata.csv")  # INSERT PATH TO INITIAL METADATA CSV HERE

# output file path
output_csv_path = "/path/to/manually/corrected/FILL_INFO_metadata.csv"  # INSERT PATH TO PROCESSED METADATA CSV HERE
output_pkl_path = "/path/to/manually/corrected/FILL_INFO_metadata.pkl"  # INSERT PATH TO PROCESSED METADATA PKL HERE

# Step 1: Remove empty rows
edited_df = cf.remove_empty_rows(edited_df)

# Step 2: Normalize participant names
edited_df = cf.normalize_names(edited_df)
# # Display unique participant names for verification
# print("Unique participant names after normalization:")
# display(edited_df['participant_name'].unique())

In [ ]:
# Step 3: Align participant IDs
edited_df = cf.align_ids(edited_df)

# Step 4: Group and display multiple IDs per participant name
id_groupby_name = cf.group_ids_by_name(edited_df)
multiple_ids = id_groupby_name[id_groupby_name['participant_id'].apply(len) > 1]

# Store validation results
validation_results = []

for idx, row in multiple_ids.iterrows():
    name = row['participant_name']
    id_set = row['participant_id']
    print(f"Participant Name: {name}")
    
    for id_val in id_set:
        is_valid = vid.CheckID(id_val)
        print(f"  ID: {id_val}, Valid: {is_valid}")
        
        # Save to list of dicts
        validation_results.append({
            "participant_name": name,
            "participant_id": id_val,
            "valid_id": is_valid
        })

# Convert to a DataFrame if you want
validation_df = pd.DataFrame(validation_results)

    

In [ ]:
# Manually replace incorrect IDs by calling replace_values
edited_df = cf.replace_values(edited_df, 'participant_id',  '123456789', 'participant_name', 'Alice')

# test the replace of IDs worked
id_groupby_name = cf.group_ids_by_name(edited_df)
multiple_ids = id_groupby_name[id_groupby_name['participant_id'].apply(len) > 1]

In [ ]:
# Step 5: Check sex inconsistencies per participant_id
id_sex_name_grouped = cf.group_sex_by_id(edited_df)
multiple_sex = id_sex_name_grouped[id_sex_name_grouped['sex'].apply(len) > 1]
print("Participants with multiple sex entries:")
display(multiple_sex)

# Manually replace sex in the DataFrame using replace_values as needed

In [ ]:
edited_df = cf.replace_values(edited_df, 'sex', 'M', 'participant_name', 'Alice')


# test the replace worked
id_sex_name_grouped = cf.group_sex_by_id(edited_df)
multiple_sex = id_sex_name_grouped[id_sex_name_grouped['sex'].apply(len) > 1]
print("Participants with multiple sex entries:")
display(multiple_sex)

After aligning names, IDs and sex, load the new subjects to subj_map.json

In [ ]:
# Step 6: Manage subj_map.json as the master subject list
json_file_path = "/home/gaia/Projects/legacy_data/my_master/subj_map.json"
subj_map = cf.load_subj_map(json_file_path)

subj_map, conflicts_df = cf.add_subjects_from_df(
    subj_map,
    edited_df['participant_name'],
    edited_df['participant_id'],
    edited_df['sex']
)

In [ ]:
# remove duplicates from conflicts_df
conflicts_df = conflicts_df.drop_duplicates(subset=['existing_id'])

# manually fix conflicts from conflicts_df - align df to subj_map (what's in subj_map is the correct info)
edited_df = cf.replace_values(edited_df, 'participant_name', 'Bob', 'participant_id', '123456789')
edited_df = cf.replace_values(edited_df, 'sex', 'F', 'participant_id', '123456789')

# Add or update subjects from the cleaned df
subj_map, conflicts_df = cf.add_subjects_from_df(
    subj_map,
    edited_df['participant_name'],
    edited_df['participant_id'],
    edited_df['sex']
)

# remove duplicates from conflicts_df
conflicts_df = conflicts_df.drop_duplicates(subset=['existing_id'])


In [ ]:
list_of_unique_names = edited_df['participant_name'].unique().tolist()

In [ ]:
# un-comment when ready to save

# # Save the updated map
# cf.save_subj_map(subj_map, json_file_path)

In [ ]:
# Step 7: Update the DataFrame with unique IDs and sex from subj_map
edited_df['unique_id'] = edited_df['participant_name'].apply(
    lambda n: next((uid for uid, sub in subj_map.items() if sub['name'] == n), None)
)
edited_df['sex'] = edited_df['unique_id'].apply(
    lambda uid: subj_map[uid]['sex'] if uid in subj_map else None
)

In [ ]:
# Step 8: Remove identifying columns for anonymization
edited_df = edited_df.drop(columns=['participant_name', 'participant_id'])

In [ ]:
from tqdm.auto import tqdm
from tqdm.autonotebook import tqdm as notebook_tqdm
import legacy_pipe.cleaning_funcs as cf



# Convert ages to years
edited_df['age_in_years'] = edited_df['age_at_scan'].apply(cf.convert_age_to_years)

# Calculate missing critical info (age_in_years, dob, scan_date)

# column to document what info was calculated
edited_df["estimated_critical_info"] = pd.NA

# Convert 'scan_date' and 'dob' to integer and skip NA values
edited_df['scan_date'] = pd.to_numeric(edited_df['scan_date'], errors='coerce')
edited_df['dob']       = pd.to_numeric(edited_df['dob'], errors='coerce')

# Convert float → integer without .0
edited_df['scan_date'] = edited_df['scan_date'].astype('Int64')
edited_df['dob']       = edited_df['dob'].astype('Int64')

# Apply the function to each row
tqdm.pandas()
edited_df = cf.calculate_age_and_critical_info_vectorized(edited_df)

# extract birth_year based of dob (first 4 characters of dob)
edited_df['birth_year'] = (
    edited_df['dob']
    .astype("string")
    .str.slice(0, 4)
    .astype("Int64")
)

# # # Unify multiple DOBs per subject to mean DOB
# # edited_df = cf.unify_multiple_dobs(edited_df)

# Save updated dataframe
edited_df.to_csv(output_csv_path, index=False)

# save as pickle too
edited_df.to_pickle(output_pkl_path)

In [ ]:
print(f"columns in final dataframe: {edited_df.columns.tolist()}")